In [1]:
import pandas as pd
import re
from IPython.display import display, HTML


In [5]:
df = pd.read_csv(r'C:\Users\joly-\Github\HUMAN\tweede_kamer\combined_tk_AI_bert_trial_567.csv')
df.head()

,Unnamed: 0,title,year,ai_related,company_hits,body,matched_keywords_all,type,persons,orgs,countries,processed,nouns,adjectives,verbs,topic,probability
0,0,Investeren in Perspectief (Beleidsnota 2018),2018,yes,"['adyen', 'google', 'x']",Ook biedt de agenda kansen aan het bedrijfslev...,"['adyen', 'google', 'x', 'kunstmatige intellig...",beleidsnota,"['Booking', 'Reinout van den Bergh']","['FMO', '42,8%', 'Google', '41,4%', 'Regeerakk...","['Azië', 'Noord-Amerika', 'Midden-Oosten', 'La...",agenda kans bedrijfsleven oplossing digitalise...,agenda kans bedrijfsleven oplossing digitalise...,innovatief traditioneel groot nieuw universeel...,bieden kennen zet ontstaan ontwikkelen gaan zo...,1,0.279240
1,1,Nota Defensie Industrie Strategie,2018,yes,[],Nederland wil zelf aan militaire kennisontwikk...,"['ai', 'artificiële intelligentie', 'drones', ...",beleidsnota,['Meeontwikkelen'],"['GEWENST', 'Energy Wapens', 'Network', 'Human...","['meeontwikkelen', 'Materiel', 'Nederland']",kennisontwikkeling analyse kennis prestatie aa...,kennisontwikkeling analyse kennis prestatie ke...,militair militair artificieel electromagnetisc...,blijven blijven maken ontwikkelen verbeteren w...,1,0.413294
2,2,Nationale Strategie Digitaal Erfgoed 2021-2024,2021,yes,['google'],Zo wordt in de vernieuwde strategie nu ook de ...,"['google', 'ai', 'algoritmes', 'artificiële in...",beleidsnota,"['Digi-', 'B. I']","['CLARIAH', 'Nederlands Dans Theater', 'Google...",['taalverwerkingstoepassingen'],strategie elatie erfgoed kunstensector industr...,strategie elatie erfgoed kunstensector industr...,creatief nieuw breed nieuw artificieel individ...,vernieuwen leggen bieden geven verkenen realis...,1,0.220815
3,3,Beslisnota's bij de Kamerbrief over toekomstig...,2021,yes,['x'],25 januari\n37 9-2-2023 Nota-StasGB-Memo box 3...,"['x', 'ai']",beleidsnota,"['Bl^w', 'voordit', 'Maria 10', 'qaan', 'budge...","['Ic', 'wuo', 'HVP', '03 rendementsklasse', 'T...","['Eengenotsrechttegeneen', 'Septemberbriefover...",vierhoek bespreking box d.d. uitstel stelsel b...,vierhoek bespreking box d.d. uitstel stelsel b...,politiek nota-stasfb-nota nota-min-memo toekom...,onroerend bespreken onderzoeken bestaan inform...,2,0.127124
4,4,Beslisnota bij Kamerbrief over aanpak belastin...,2021,yes,['x'],Dit betreft de nota’s in de onderstaande tabel...,"['x', 'ai']",beleidsnota,"['datvoor de huidige', 'Stas T', 'Teonderteken...","['Unie', 'Europese Commissie', 'afwikkelfonds'...","['Kopleaan', 'veriagen', 'Eventueei', 'Middenb...",nota tabel titel aanpak verband vervolgnota ve...,nota tabel titel aanpak verband vervolgnota ve...,onderstaand herziene abusievelijk vertrouwelij...,betreffen belastingschulden belastingschulen b...,2,0.204630


In [4]:
df.shape

(298, 22)

In [6]:
# -----------------------------
# SETTINGS
# -----------------------------
topic_id = 12 # choose topic
n_examples = 27  # number of rows to display
text_col = "body"  # or "text_full", etc.

# -----------------------------
# YOUR KEYWORDS
# -----------------------------
keywords = (
    "kunstmatige intelligentie|artificial intelligence|artificiële intelligentie|AI|generatieve AI|"
    "generatieve kunstmatige intelligentie|generatieve artificiële intelligentie|"
    "machine learning|machinaal leren|diep leren|deep learning|neurale netwerken|"
    "large language model|grote taalmodel*|LLM|chatbot*|GPT|ChatGPT|Bard|Claude|"
    "Gemini|mistral|perplexity|ollama|LLaMA|openai|anthropic|midjourney|hugging face|"
    "slimme algoritme*|automatische besluitvorming|automatisch beslissysteem|"
    "algoritmische besluitvorming|algoritme*|cognitieve technologie*|AI-technologie*|"
    "AI-systeem*|AI-toepassing*|AI-model*|spraakherkenning|beeldherkenning|"
    "computer vision|natuurlijke taalverwerking|natural language processing|NLP|robot|drones|drone|grok|xai|deepmind|azure"
)

_COMPANY_NAMES = [
    "NVIDIA", "Apple", "Microsoft", "Google", "Alphabet",
    "Meta Platforms", "Facebook", "Tesla", "Oracle",
    "Palantir", "IBM", "Adobe", "Cambricon Technologies",
    "CoreWeave", "Fermi Inc", "Dynatrace", "Tempus AI",
    "SenseTime", "Mobileye", "Aurora Innovation", "UiPath",
    "SoundHound AI", "ASML", "NXP Semiconductors",
    "BE Semiconductor Industries", "ASM International",
    "Adyen", "Just Eat Takeaway", "Booking.com", "Mollie",
    "Picnic", "TomTom", "Swapfiets", "TKH Group",
    "Ordina", "Nedap", "CM.com", "ICT Group",
    "Neways Electronics", "Ctac", "Photon Energy",
    "Almunda Professionals", "Samsung", "Huawei",
    "Sony", "LG", "Baidu", "Tencent",
    "Alibaba", "Douyin", "Cloudflare",
    "Snowflake", "Docker", "Red Hat",
    "Uber", "Bolt", "Grab", "Epic Games",
    "Unity", "Discord", "Twitter", "X"
]

# -----------------------------
# BUILD REGEX
# -----------------------------

def wildcard_to_regex(pattern):
    return pattern.replace("*", r"\w*")

keyword_patterns = [wildcard_to_regex(k) for k in keywords.split("|")]
company_patterns = [re.escape(c) for c in _COMPANY_NAMES]

all_patterns = keyword_patterns + company_patterns

regex = re.compile(r"\b(" + "|".join(all_patterns) + r")\b", re.IGNORECASE)

# -----------------------------
# HIGHLIGHT FUNCTION
# -----------------------------

def highlight_text(text):
    if pd.isna(text):
        return ""
    
    def repl(match):
        return f"<span style='color:red; font-weight:bold; font-size:120%;'>{match.group(0)}</span>"
    
    return regex.sub(repl, text)

# -----------------------------
# FILTER + SAMPLE
# -----------------------------

df_topic = df[df["topic"] == topic_id].copy()
df_sample = df_topic.sample(n=min(n_examples, len(df_topic)), random_state=42)

# -----------------------------
# DISPLAY
# -----------------------------

for i, row in df_sample.iterrows():
    title = row.get("title", "")
    text = row.get(text_col, "")

    # also show filename, program and year  
    filename = row.get("filename", "")
    program = row.get("program", "")
    year = row.get("year", "")
    meta_info = f"<small><em>{filename} | {program} | {year}</em></small>"
    
    
    html = f"""
    <div style="margin-bottom:30px;">
        {meta_info}
        <h4>{highlight_text(title)}</h4>
        <p>{highlight_text(text)}</p>
    </div>
    """
    
    display(HTML(html))

In [ ]:

# optional: save to Excel
# df_topic.to_excel(f"topic_{topic_id}_subset.xlsx", index=False)